# Original airline data

Обучаем вспомогательную модель на исходных 129 880 строках. Ее прогноз добавим как один новый признак к нашей Route TE модели.

In [1]:
from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"
oof_dir = project_root / "artifacts" / "oof"
test_predictions_dir = project_root / "artifacts" / "test_predictions"
metrics_dir = project_root / "artifacts" / "metrics"

for directory in [oof_dir, test_predictions_dir, metrics_dir]:
    directory.mkdir(parents=True, exist_ok=True)

train_df = pd.read_parquet(processed_data_dir / "train_with_folds.parquet")
test_df = pd.read_parquet(processed_data_dir / "test_clean.parquet")
original_df = pd.read_csv(raw_data_dir / "airline_passenger_satisfaction.csv")

print("competition train:", train_df.shape)
print("competition test:", test_df.shape)
print("original data:", original_df.shape)

competition train: (699635, 25)
competition test: (299844, 23)
original data: (129880, 24)


## Приводим исходные данные к нашей схеме

Здесь меняются только названия столбцов и категорий. `In-flight Service` удаляем: такого признака нет в competition train и test.

In [3]:
column_names = {
    "Departure Delay": "Departure Delay in Minutes",
    "Arrival Delay": "Arrival Delay in Minutes",
    "Departure and Arrival Time Convenience": "Departure/Arrival time convenient",
    "Ease of Online Booking": "Ease of Online booking",
    "Check-in Service": "Checkin service",
    "Online Boarding": "Online boarding",
    "Gate Location": "Gate location",
    "On-board Service": "On-board service",
    "Seat Comfort": "Seat comfort",
    "Leg Room Service": "Leg room service",
    "Food and Drink": "Food and drink",
    "In-flight Wifi Service": "Inflight wifi service",
    "In-flight Entertainment": "Inflight entertainment",
    "Baggage Handling": "Baggage handling",
}

original_df = original_df.rename(columns=column_names)

original_df["Gender"] = original_df["Gender"].map({
    "Female": 0,
    "Male": 1,
})

original_df["Customer Type"] = original_df["Customer Type"].map({
    "First-time": 0,
    "Returning": 1,
})

original_df["Type of Travel"] = original_df["Type of Travel"].map({
    "Personal": 0,
    "Business": 1,
})

original_df["Class"] = original_df["Class"].replace({
    "Economy": "Eco",
    "Economy Plus": "Eco Plus",
})

original_df["Arrival Delay Missing"] = (
    original_df["Arrival Delay in Minutes"].isna().astype("int8")
)

original_df["satisfaction"] = (
    original_df["Satisfaction"] == "Satisfied"
).astype("int8")

feature_columns = [
    column
    for column in train_df.columns
    if column not in ["id", "satisfaction", "fold"]
]

original_df = original_df[feature_columns + ["satisfaction"]].copy()

assert original_df[feature_columns].isna().sum().drop(
    "Arrival Delay in Minutes"
).sum() == 0

print("features:", len(feature_columns))
print("original prepared:", original_df.shape)
print("target mean:", original_df["satisfaction"].mean())
print("classes:", sorted(original_df["Class"].unique()))

features: 22
original prepared: (129880, 23)
target mean: 0.43446258084385586
classes: ['Business', 'Eco', 'Eco Plus']


## Вспомогательная модель

Она видит только исходный датасет. Затем пять моделей прогнозируют вероятности для competition train и test. Competition-метки здесь не используются, поэтому утечки нет.

In [4]:
source_x = original_df[feature_columns]
source_y = original_df["satisfaction"]
competition_x = train_df[feature_columns]
competition_test_x = test_df[feature_columns]

source_oof = np.zeros(len(original_df), dtype="float32")
competition_probability = np.zeros(len(train_df), dtype="float32")
test_probability = np.zeros(len(test_df), dtype="float32")
source_scores = []

source_folds = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

source_parameters = {
    "n_estimators": 2500,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "min_child_samples": 30,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "n_jobs": 4,
    "verbosity": -1,
}

class_categories = sorted(
    set(source_x["Class"].dropna().unique())
    | set(competition_x["Class"].dropna().unique())
    | set(competition_test_x["Class"].dropna().unique())
)

for fold, (train_index, valid_index) in enumerate(
    source_folds.split(source_x, source_y)
):
    print(f"\nsource fold {fold}")

    x_train = source_x.iloc[train_index].copy()
    x_valid = source_x.iloc[valid_index].copy()
    x_competition = competition_x.copy()
    x_test = competition_test_x.copy()

    y_train = source_y.iloc[train_index]
    y_valid = source_y.iloc[valid_index]

    arrival_median = x_train["Arrival Delay in Minutes"].median()

    for frame in [x_train, x_valid, x_competition, x_test]:
        frame["Arrival Delay in Minutes"] = (
            frame["Arrival Delay in Minutes"].fillna(arrival_median)
        )
        frame["Class"] = pd.Categorical(
            frame["Class"],
            categories=class_categories,
        )

    model = lgb.LGBMClassifier(
        **source_parameters,
        random_state=42 + fold,
    )

    model.fit(
        x_train,
        y_train,
        eval_set=[(x_valid, y_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(100),
        ],
    )

    valid_prediction = model.predict_proba(
        x_valid,
        num_iteration=model.best_iteration_,
    )[:, 1]

    source_oof[valid_index] = valid_prediction
    competition_probability += model.predict_proba(
        x_competition,
        num_iteration=model.best_iteration_,
    )[:, 1].astype("float32") / 5

    test_probability += model.predict_proba(
        x_test,
        num_iteration=model.best_iteration_,
    )[:, 1].astype("float32") / 5

    fold_auc = roc_auc_score(y_valid, valid_prediction)
    source_scores.append(float(fold_auc))

    print("source AUC:", fold_auc)
    print("best iteration:", model.best_iteration_)

    del model
    del x_train, x_valid, x_competition, x_test
    del y_train, y_valid, valid_prediction
    gc.collect()


source fold 0


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.991413	valid_0's binary_logloss: 0.140955
[200]	valid_0's auc: 0.994009	valid_0's binary_logloss: 0.0994093
[300]	valid_0's auc: 0.994671	valid_0's binary_logloss: 0.0918243
[400]	valid_0's auc: 0.994924	valid_0's binary_logloss: 0.0887427
[500]	valid_0's auc: 0.995047	valid_0's binary_logloss: 0.0872383
[600]	valid_0's auc: 0.995123	valid_0's binary_logloss: 0.0863532
[700]	valid_0's auc: 0.995162	valid_0's binary_logloss: 0.0857475
[800]	valid_0's auc: 0.995203	valid_0's binary_logloss: 0.0850849
[900]	valid_0's auc: 0.995224	valid_0's binary_logloss: 0.0846585
[1000]	valid_0's auc: 0.99524	valid_0's binary_logloss: 0.0844092
[1100]	valid_0's auc: 0.99525	valid_0's binary_logloss: 0.0842287
Early stopping, best iteration is:
[1065]	valid_0's auc: 0.995258	valid_0's binary_logloss: 0.0842147
source AUC: 0.995257574081009
best iteration: 1065

source fold 1


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.990797	valid_0's binary_logloss: 0.143209
[200]	valid_0's auc: 0.993502	valid_0's binary_logloss: 0.103112
[300]	valid_0's auc: 0.994212	valid_0's binary_logloss: 0.0952868
[400]	valid_0's auc: 0.994528	valid_0's binary_logloss: 0.0919995
[500]	valid_0's auc: 0.994656	valid_0's binary_logloss: 0.0906071
[600]	valid_0's auc: 0.994748	valid_0's binary_logloss: 0.0894831
[700]	valid_0's auc: 0.994806	valid_0's binary_logloss: 0.0887705
[800]	valid_0's auc: 0.994841	valid_0's binary_logloss: 0.0882882
[900]	valid_0's auc: 0.994838	valid_0's binary_logloss: 0.0880833
Early stopping, best iteration is:
[854]	valid_0's auc: 0.994852	valid_0's binary_logloss: 0.0880583
source AUC: 0.9948518449186307
best iteration: 854

source fold 2


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.991945	valid_0's binary_logloss: 0.137154
[200]	valid_0's auc: 0.994333	valid_0's binary_logloss: 0.0972306
[300]	valid_0's auc: 0.994906	valid_0's binary_logloss: 0.0903146
[400]	valid_0's auc: 0.995154	valid_0's binary_logloss: 0.0870574
[500]	valid_0's auc: 0.995248	valid_0's binary_logloss: 0.0855465
[600]	valid_0's auc: 0.995298	valid_0's binary_logloss: 0.0847216
[700]	valid_0's auc: 0.995347	valid_0's binary_logloss: 0.0839992
[800]	valid_0's auc: 0.995349	valid_0's binary_logloss: 0.0837237
Early stopping, best iteration is:
[713]	valid_0's auc: 0.995351	valid_0's binary_logloss: 0.0839434
source AUC: 0.9953509272559109
best iteration: 713

source fold 3


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.991185	valid_0's binary_logloss: 0.142332
[200]	valid_0's auc: 0.993663	valid_0's binary_logloss: 0.102489
[300]	valid_0's auc: 0.994388	valid_0's binary_logloss: 0.0944533
[400]	valid_0's auc: 0.994599	valid_0's binary_logloss: 0.0919481
[500]	valid_0's auc: 0.994708	valid_0's binary_logloss: 0.0905094
[600]	valid_0's auc: 0.994765	valid_0's binary_logloss: 0.0897162
[700]	valid_0's auc: 0.994823	valid_0's binary_logloss: 0.0890181
[800]	valid_0's auc: 0.994884	valid_0's binary_logloss: 0.0882805
[900]	valid_0's auc: 0.994891	valid_0's binary_logloss: 0.0880002
[1000]	valid_0's auc: 0.994908	valid_0's binary_logloss: 0.0877029
[1100]	valid_0's auc: 0.994913	valid_0's binary_logloss: 0.0875709
[1200]	valid_0's auc: 0.994914	valid_0's binary_logloss: 0.0874821
[1300]	valid_0's auc: 0.994915	valid_0's binary_logloss: 0.087412
Early stopping, best iteration is:
[1208]	valid_0's auc: 0.994919	valid_0's bin

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.990841	valid_0's binary_logloss: 0.142306
[200]	valid_0's auc: 0.993558	valid_0's binary_logloss: 0.102181
[300]	valid_0's auc: 0.994279	valid_0's binary_logloss: 0.0941732
[400]	valid_0's auc: 0.994572	valid_0's binary_logloss: 0.091111
[500]	valid_0's auc: 0.994719	valid_0's binary_logloss: 0.0894508
[600]	valid_0's auc: 0.994779	valid_0's binary_logloss: 0.0887101
[700]	valid_0's auc: 0.994816	valid_0's binary_logloss: 0.0882142
[800]	valid_0's auc: 0.994842	valid_0's binary_logloss: 0.0878567
Early stopping, best iteration is:
[784]	valid_0's auc: 0.994848	valid_0's binary_logloss: 0.087834
source AUC: 0.9948475535573812
best iteration: 784


In [5]:
source_auc = roc_auc_score(source_y, source_oof)

np.save(oof_dir / "original_source_oof.npy", source_oof)
np.save(oof_dir / "original_probability_train.npy", competition_probability)
np.save(
    test_predictions_dir / "original_probability_test.npy",
    test_probability,
)

source_metrics = {
    "fold_auc": source_scores,
    "oof_auc": float(source_auc),
    "parameters": source_parameters,
}

with open(metrics_dir / "original_source_model.json", "w", encoding="utf-8") as file:
    json.dump(source_metrics, file, indent=4, ensure_ascii=False)

print("source fold AUC:", source_scores)
print("source OOF AUC:", source_auc)
print("train probability range:", competition_probability.min(), competition_probability.max())
print("test probability range:", test_probability.min(), test_probability.max())

source fold AUC: [0.995257574081009, 0.9948518449186307, 0.9953509272559109, 0.9949191254500991, 0.9948475535573812]
source OOF AUC: 0.995040203342028
train probability range: 2.7027765e-05 0.9999915
test probability range: 3.286764e-05 0.9999915


## Проверяем новый признак на competition CV

Пока запускаем только fold 0. Старый результат на нём — `0.9603531996`. Если новый AUC выше, запустим остальные фолды.

In [8]:
train_df["Original Probability"] = competition_probability
test_df["Original Probability"] = test_probability

route_column = "Flight Distance"
target = "satisfaction"
folds_to_run = [1, 2, 3, 4]

model_features = [
    column
    for column in train_df.columns
    if column not in ["id", target, "fold"]
]

route_scores = {
    0: 0.9603531995583718,
    1: 0.9588368924864543,
    2: 0.9608820613035495,
    3: 0.9603249147336925,
    4: 0.9602052661607016,
}

competition_parameters = {
    "n_estimators": 2000,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "n_jobs": 4,
    "verbosity": -1,
}

print("features:", len(model_features))
print("folds:", folds_to_run)

features: 23
folds: [1, 2, 3, 4]


In [9]:
fold_metrics = {}

for fold in folds_to_run:
    print(f"\ncompetition fold {fold}")

    train_mask = train_df["fold"] != fold
    valid_mask = train_df["fold"] == fold

    x_train = train_df.loc[train_mask, model_features].copy()
    x_valid = train_df.loc[valid_mask, model_features].copy()
    x_test = test_df[model_features].copy()

    y_train = train_df.loc[train_mask, target]
    y_valid = train_df.loc[valid_mask, target]

    inner_folds = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42 + fold,
    )

    encoder = TargetEncoder(
        target_type="binary",
        smooth="auto",
        cv=inner_folds,
    )

    x_train["Flight Distance TE"] = encoder.fit_transform(
        x_train[[route_column]],
        y_train,
    ).ravel()

    x_valid["Flight Distance TE"] = encoder.transform(
        x_valid[[route_column]]
    ).ravel()

    x_test["Flight Distance TE"] = encoder.transform(
        x_test[[route_column]]
    ).ravel()

    arrival_median = x_train["Arrival Delay in Minutes"].median()
    class_categories = sorted(
        set(x_train["Class"].dropna().unique())
        | set(x_valid["Class"].dropna().unique())
        | set(x_test["Class"].dropna().unique())
    )

    for frame in [x_train, x_valid, x_test]:
        frame["Arrival Delay in Minutes"] = (
            frame["Arrival Delay in Minutes"].fillna(arrival_median)
        )
        frame["Class"] = pd.Categorical(
            frame["Class"],
            categories=class_categories,
        )

    model = lgb.LGBMClassifier(
        **competition_parameters,
        random_state=42 + fold,
    )

    model.fit(
        x_train,
        y_train,
        eval_set=[(x_valid, y_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(100),
        ],
    )

    valid_prediction = model.predict_proba(
        x_valid,
        num_iteration=model.best_iteration_,
    )[:, 1].astype("float32")

    test_prediction = model.predict_proba(
        x_test,
        num_iteration=model.best_iteration_,
    )[:, 1].astype("float32")

    fold_auc = roc_auc_score(y_valid, valid_prediction)
    difference = fold_auc - route_scores[fold]

    np.save(oof_dir / f"original_route_fold_{fold}.npy", valid_prediction)
    np.save(
        test_predictions_dir / f"original_route_fold_{fold}.npy",
        test_prediction,
    )

    fold_metrics[str(fold)] = {
        "auc": float(fold_auc),
        "route_auc": float(route_scores[fold]),
        "difference": float(difference),
        "best_iteration": int(model.best_iteration_),
    }

    with open(
        metrics_dir / "original_route_folds.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(fold_metrics, file, indent=4, ensure_ascii=False)

    print("AUC:", fold_auc)
    print("old Route TE:", route_scores[fold])
    print("difference:", difference)
    print("best iteration:", model.best_iteration_)

    del model, encoder
    del x_train, x_valid, x_test
    del y_train, y_valid
    del valid_prediction, test_prediction
    gc.collect()


competition fold 1


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.957531	valid_0's binary_logloss: 0.234315
[200]	valid_0's auc: 0.958694	valid_0's binary_logloss: 0.227871
[300]	valid_0's auc: 0.959053	valid_0's binary_logloss: 0.227267
[400]	valid_0's auc: 0.959139	valid_0's binary_logloss: 0.227109
[500]	valid_0's auc: 0.959242	valid_0's binary_logloss: 0.226971
[600]	valid_0's auc: 0.959267	valid_0's binary_logloss: 0.226903
Early stopping, best iteration is:
[574]	valid_0's auc: 0.959282	valid_0's binary_logloss: 0.226899
AUC: 0.9592822337505565
old Route TE: 0.9588368924864543
difference: 0.0004453412641022281
best iteration: 574

competition fold 2


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.959537	valid_0's binary_logloss: 0.229831
[200]	valid_0's auc: 0.960616	valid_0's binary_logloss: 0.223
[300]	valid_0's auc: 0.960927	valid_0's binary_logloss: 0.222333
[400]	valid_0's auc: 0.961051	valid_0's binary_logloss: 0.222084
[500]	valid_0's auc: 0.961114	valid_0's binary_logloss: 0.221953
[600]	valid_0's auc: 0.961151	valid_0's binary_logloss: 0.221885
[700]	valid_0's auc: 0.961182	valid_0's binary_logloss: 0.221807
[800]	valid_0's auc: 0.961178	valid_0's binary_logloss: 0.221809
[900]	valid_0's auc: 0.961205	valid_0's binary_logloss: 0.221759
[1000]	valid_0's auc: 0.961183	valid_0's binary_logloss: 0.221761
Early stopping, best iteration is:
[929]	valid_0's auc: 0.961211	valid_0's binary_logloss: 0.221747
AUC: 0.9612112984983803
old Route TE: 0.9608820613035495
difference: 0.0003292371948308004
best iteration: 929

competition fold 3


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.958909	valid_0's binary_logloss: 0.231185
[200]	valid_0's auc: 0.960158	valid_0's binary_logloss: 0.224701
[300]	valid_0's auc: 0.960471	valid_0's binary_logloss: 0.223986
[400]	valid_0's auc: 0.960541	valid_0's binary_logloss: 0.223754
[500]	valid_0's auc: 0.960613	valid_0's binary_logloss: 0.223596
[600]	valid_0's auc: 0.960613	valid_0's binary_logloss: 0.223532
Early stopping, best iteration is:
[535]	valid_0's auc: 0.960627	valid_0's binary_logloss: 0.223559
AUC: 0.960626979745212
old Route TE: 0.9603249147336925
difference: 0.0003020650115195256
best iteration: 535

competition fold 4


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.95918	valid_0's binary_logloss: 0.232081
[200]	valid_0's auc: 0.960155	valid_0's binary_logloss: 0.225262
[300]	valid_0's auc: 0.960463	valid_0's binary_logloss: 0.224621
[400]	valid_0's auc: 0.960607	valid_0's binary_logloss: 0.224356
[500]	valid_0's auc: 0.960666	valid_0's binary_logloss: 0.224231
[600]	valid_0's auc: 0.960692	valid_0's binary_logloss: 0.224179
Early stopping, best iteration is:
[575]	valid_0's auc: 0.96071	valid_0's binary_logloss: 0.22417
AUC: 0.9607096688639081
old Route TE: 0.9602052661607016
difference: 0.0005044027032065079
best iteration: 575


## Собираем пять фолдов

Валидационные прогнозы возвращаем на их исходные позиции, а пять прогнозов test усредняем.

In [10]:
submissions_dir = project_root / "submissions"
submissions_dir.mkdir(parents=True, exist_ok=True)

sample_submission = pd.read_csv(raw_data_dir / "sample_submission.csv")
final_oof = np.zeros(len(train_df), dtype="float32")
test_fold_predictions = []
final_fold_scores = []

for fold in range(5):
    oof_path = oof_dir / f"original_route_fold_{fold}.npy"
    test_path = test_predictions_dir / f"original_route_fold_{fold}.npy"

    if not oof_path.exists() or not test_path.exists():
        raise FileNotFoundError(f"Missing fold {fold}")

    valid_mask = train_df["fold"] == fold
    fold_prediction = np.load(oof_path)
    test_fold_prediction = np.load(test_path)

    final_oof[valid_mask] = fold_prediction
    test_fold_predictions.append(test_fold_prediction)

    fold_auc = roc_auc_score(
        train_df.loc[valid_mask, target],
        fold_prediction,
    )
    final_fold_scores.append(float(fold_auc))

final_test_prediction = np.mean(
    test_fold_predictions,
    axis=0,
).astype("float32")

final_auc = roc_auc_score(train_df[target], final_oof)

np.save(oof_dir / "original_route.npy", final_oof)
np.save(test_predictions_dir / "original_route.npy", final_test_prediction)

final_metrics = {
    "fold_auc": final_fold_scores,
    "mean_fold_auc": float(np.mean(final_fold_scores)),
    "oof_auc": float(final_auc),
    "source_oof_auc": float(source_auc),
    "parameters": competition_parameters,
}

with open(metrics_dir / "original_route.json", "w", encoding="utf-8") as file:
    json.dump(final_metrics, file, indent=4, ensure_ascii=False)

submission = sample_submission.copy()
submission[target] = final_test_prediction
submission_path = submissions_dir / "original_route_submission.csv"
submission.to_csv(submission_path, index=False)

print("fold AUC:", final_fold_scores)
print("mean fold AUC:", np.mean(final_fold_scores))
print("OOF AUC:", final_auc)
print("saved:", submission_path)
submission.head()

fold AUC: [0.9605212904729112, 0.9592822337505565, 0.9612112984983803, 0.960626979745212, 0.9607096688639081]
mean fold AUC: 0.9604702942661936
OOF AUC: 0.9604621524485576
saved: c:\Users\User\Desktop\Airline Satisfaction\submissions\original_route_submission.csv


,id,satisfaction
0,699635,0.227141
1,699636,0.817710
2,699637,0.932096
3,699638,0.076701
4,699639,0.052986


После fold 0 сравниваем `AUC` и `old Route TE`. Остальные фолды запускаем только при положительной разнице.